In [1]:
import torch

In [5]:
torch.zeros(4, 8, 16).dim()

3

In [416]:
import math

import torch
from einops import einsum


class Linear(torch.nn.Module):
    weights: torch.Tensor

    def __init__(self, in_features: int, out_features: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.weights = torch.nn.parameter.Parameter(torch.empty(out_features, in_features, device=device, dtype=dtype))
        std = math.sqrt(2 / (in_features + out_features))
        torch.nn.init.trunc_normal_(tensor=self.weights, mean=0, std=std, a=-3 * std, b=3 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return einsum(x, self.weights, "... in, out in -> ... out")

In [417]:
linear = Linear(1024, 1024)
linear.forward(torch.randn(1024, 1024))

list(linear.state_dict().keys())

['weights']

In [418]:
import torch
import math
from einops import einsum

class Embedding(torch.nn.Module):

    embeddings: torch.Tensor # (num_embeddings, embedding_dim)

    def __init__(self, num_embeddings: int, embedding_dim: int, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.embeddings = torch.nn.parameter.Parameter(torch.empty(num_embeddings, embedding_dim, device=device, dtype=dtype))
        torch.nn.init.trunc_normal_(tensor=self.embeddings, mean=0, std=1, a=-3, b=3)

    # token_ids: torch.LongTensor (batch_size, sequence_length)
    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings[token_ids]

In [419]:
embedding = Embedding(256, 1024)
embedding.forward(torch.randint(0, 256, (2, 5)))

tensor([[[-0.0482, -0.2582, -1.0374,  ...,  1.7179,  1.9145, -2.2027],
         [-0.1800,  2.1938, -1.3332,  ...,  1.0064, -0.5730, -1.0454],
         [-1.8975,  0.2923,  2.0056,  ...,  0.9881, -0.3819, -1.5775],
         [-0.6153,  0.1556,  2.1111,  ...,  0.4297,  0.5600,  2.0212],
         [-0.6011, -1.4146, -0.8549,  ...,  0.2481,  0.1182, -0.4327]],

        [[-0.9834,  0.9497,  0.1678,  ..., -0.4269,  0.5367,  0.0396],
         [-1.0469,  0.1624, -0.8158,  ..., -0.4258, -1.1824, -0.9600],
         [-0.0137,  0.7782, -0.3195,  ..., -2.0292, -0.5957,  1.3385],
         [ 0.1698,  0.2836, -1.4835,  ...,  0.3850,  1.2755, -0.5289],
         [ 0.0288, -0.3160, -0.3742,  ..., -0.1036, -0.5483, -1.5206]]],
       grad_fn=<IndexBackward0>)

In [514]:
import torch

class RMSNorm(torch.nn.Module):

    gain: torch.Tensor # (d_model, )
    eps: float
    
    def __init__(self, d_model: int, eps: float = 1e-5, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.gain = torch.nn.parameter.Parameter(torch.ones(d_model, device=device, dtype=dtype))
        self.eps = eps

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.to(torch.float32)
        x_sqrd_mean = x.pow(2).mean(dim=-1, keepdim=True) # (batch_size, sequence_length, 1)
        rms = torch.sqrt(x_sqrd_mean + self.eps) # (batch_size, sequence_length, 1)
        # (batch_size, sequence_length, d_model) * (d_model, ) / (batch_size, sequence_length, 1)
        result = x * self.gain / rms # (batch_size, sequence_length, d_model)
        return result.to(in_dtype)


In [515]:
rms_norm = RMSNorm(1024)
rms_norm.forward(torch.randn(1024))

tensor([ 1.0308, -0.3498,  0.1247,  ...,  1.1425,  0.1272, -0.8842],
       grad_fn=<DivBackward0>)

In [486]:
import torch

class SwiGLU(torch.nn.Module):
    w_1: Linear # (d_ff, d_model)
    w_2: Linear # (d_model, d_ff)
    w_3: Linear # (d_ff, d_model)

    def __init__(self, d_model: int, d_ff: int | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        if d_ff is None:
            d_ff = round(8 * d_model / 3 / 64) * 64
        self.w_1 = Linear(d_model, d_ff, device, dtype)
        self.w_2 = Linear(d_ff, d_model, device, dtype)
        self.w_3 = Linear(d_model, d_ff, device, dtype)

    # x (batch_size, sequence_length, d_model)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w_2(self._silu(self.w_1(x)) * self.w_3(x))

    def _silu(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.sigmoid(x)

In [487]:
d_model: int = 1024
swiglu = SwiGLU(d_model)
swiglu.forward(torch.randn(256, d_model))

tensor([[ 0.2433,  0.2949, -0.2479,  ...,  0.7514,  0.0520, -0.0558],
        [ 0.0971,  0.4673, -0.3477,  ...,  0.0507,  0.6041,  0.2216],
        [ 0.2029,  0.1748,  0.9017,  ..., -0.0514, -0.0229,  0.0532],
        ...,
        [ 0.3071,  0.1926,  0.6820,  ...,  0.1423, -0.1173,  0.2077],
        [ 0.0855,  0.0046, -0.5602,  ..., -0.0244, -0.3001, -0.0056],
        [-0.2647, -0.2921, -0.4038,  ...,  0.1050,  0.1413, -0.0069]],
       grad_fn=<ViewBackward0>)

In [424]:
import torch
from einops import rearrange

class RotaryPositionalEmbedding(torch.nn.Module):

    def __init__(self, theta: float, d_k: int, max_seq_len: int, device: torch.device | None = None):
        super().__init__()
        k = torch.arange(0, d_k, 2, device=device)
        positions = torch.arange(0, max_seq_len, 1, device=device).reshape(max_seq_len, 1)
        angles = positions / theta ** (k / d_k)
        self.register_buffer('sin', torch.sin(angles), persistent=False)
        self.register_buffer('cos', torch.cos(angles), persistent=False)

    # x (batch, seq_len, d_k), token_positions (batch, seq_len)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        x = rearrange(x, "... seq_len (half two) -> ... seq_len half two", two=2)
        a = x[..., 0] # (..., seq, half)
        b = x[..., 1] # (..., seq, half)
        cos = self.cos[token_positions]
        sin = self.sin[token_positions]
        a_out = a * cos - b * sin
        b_out = a * sin + b * cos
        x_out = torch.stack([a_out, b_out], dim=-1) # (..., seq_len, d_k/2 2)
        x_out = rearrange(x_out, "... seq_len half two -> ... seq_len (half two)")
        return x_out

In [425]:
d_k = 4
theta = 10000
max_seq_len = 2
x = torch.tensor([[1., 0., 1., 0.], [1., 0., 1., 0.]])
rope = RotaryPositionalEmbedding(theta, d_k, max_seq_len) # theta, d_k, max_seq_len
rope.cos.shape # seq_len, d_k/2
token_positions = torch.arange(max_seq_len)
rope.forward(x, token_positions)

tensor([[1.0000, 0.0000, 1.0000, 0.0000],
        [0.5403, 0.8415, 0.9999, 0.0100]])

In [441]:
import torch

def softmax(x: torch.Tensor, dim: int) -> torch.Tensor:
    x_max = x.max(dim=-1, keepdim=True).values
    x_norm = x.subtract(x_max)
    return x_norm.exp() / x_norm.exp().sum(dim=dim, keepdim=True)

In [442]:
x = torch.randn(2, 3)
print(x)
y = x.max(dim=-1, keepdim=True).values
print(y)
print(x.subtract(y))
print(softmax(x, -1))

tensor([[-0.4375,  0.2873,  0.3676],
        [ 0.1419,  0.7633,  1.0310]])
tensor([[0.3676],
        [1.0310]])
tensor([[-0.8050, -0.0803,  0.0000],
        [-0.8890, -0.2677,  0.0000]])
tensor([[0.1886, 0.3894, 0.4220],
        [0.1889, 0.3516, 0.4595]])


In [443]:
torch.allclose(torch.softmax(x, dim=-1), softmax(x, -1))

True

In [450]:
import torch 
from jaxtyping import Bool, Float

d_k = 4
d_v = 3
keys = 2
queries = 2

Q: Float[torch.Tensor, " ... queries d_k"] = torch.randn(keys, d_k)
K: Float[torch.Tensor, " ... keys d_k"] = torch.randn(queries, d_k)
V: Float[torch.Tensor, " ... keys d_v"] = torch.randn(keys, d_v)
mask: Bool[torch.Tensor, " ... queries keys"] | None = (torch.randn(queries, keys).uniform_() > 0.8)

In [454]:
import torch
from jaxtyping import Bool, Float
from einops import einsum
import math

def scaled_dot_product_attention(
        Q: Float[torch.Tensor, " ... queries d_k"],
        K: Float[torch.Tensor, " ... keys d_k"],
        V: Float[torch.Tensor, " ... keys d_v"],
        mask: Bool[torch.Tensor, " ... queries keys"] | None = None
) -> Float[torch.Tensor, " ... seq_len d_v"]:
    d_k = Q.shape[-1]
    qk = einsum(Q, K, "... queries d_k, ... keys d_k -> ... queries keys") / math.sqrt(d_k)
    if mask is not None:
        qk = qk.masked_fill(~mask, float("-Inf"))
    qk = softmax(qk, dim=-1)
    return einsum(qk, V, "... queries keys, ... keys d_v -> ... queries d_v")

In [455]:
scaled_dot_product_attention(Q, K, V, mask)

tensor([[    nan,     nan,     nan],
        [ 2.3111, -1.9588, -0.2809]])

In [456]:
torch.allclose(torch.nn.functional.scaled_dot_product_attention(Q, K, V), scaled_dot_product_attention(Q, K, V))

True

In [476]:
import torch
from einops import einsum, rearrange

class CasualMultiHeadSelfAttention(torch.nn.Module):

    device: torch.device | None
    d_k: int
    d_v: int
    d_model: int
    num_heads: int
    weights_q: Linear # (num_heads * d_k, d_model) = (hd_k, d_model)
    weights_k: Linear # (num_heads * d_k, d_model) = (hd_k, d_model)
    weights_v: Linear # (num_heads * d_v, d_model) = (hd_v, d_model)
    weights_o: Linear # (d_model, num_heads * d_v) = (d_model, hd_v)
    rope: RotaryPositionalEmbedding | None

    def __init__(self, d_model: int, num_heads: int, max_seq_len: int | None = None, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        if d_model % num_heads != 0:
            raise Exception("d_model must be divisible by num_heads!")
        self.device = device
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = round(d_model / num_heads)
        self.d_v = self.d_k
        self.weights_q = Linear(d_model, num_heads * self.d_k, device, dtype)
        self.weights_k = Linear(d_model, num_heads * self.d_k, device, dtype)
        self.weights_v = Linear(d_model, num_heads * self.d_v, device, dtype)
        self.weights_o = Linear(num_heads * self.d_v, d_model, device, dtype)
        if max_seq_len is not None and theta is not None:
            self.rope = RotaryPositionalEmbedding(theta, self.d_k, max_seq_len)
        else:
            self.rope = None

    # x (..., seq_len, d_model)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        seq_len = x.shape[-2]
        casual_mask = (torch.tril(torch.ones(seq_len, seq_len, device=self.device)) == 1)
        if token_positions is None:
            token_positions = torch.arange(seq_len, device=self.device)
        wq_x = self.weights_q.forward(x)  # (..., seq_len, hd_k)
        wk_x = self.weights_k.forward(x)  # (..., seq_len, hd_k)
        wv_x = self.weights_v.forward(x)  # (..., seq_len, hd_v)
        wq_x_i = rearrange(wq_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wk_x_i = rearrange(wk_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wv_x_i = rearrange(wv_x, "... seq_len (h d_v) -> ... h seq_len d_v", h=self.num_heads)  # (..., h, seq_len, d_v)
        if self.rope:
            wq_x_i = self.rope.forward(wq_x_i, token_positions)
            wk_x_i = self.rope.forward(wk_x_i, token_positions)
        result = scaled_dot_product_attention(wq_x_i, wk_x_i, wv_x_i, casual_mask)  # (..., h, seq_len, d_v)
        result = rearrange(result, "... h seq_len d_v -> ... seq_len (h d_v)")
        return self.weights_o.forward(result)


In [ ]:
import torch
from einops import einsum, rearrange

class CasualMultiHeadSelfAttentionOptimized(torch.nn.Module):

    device: torch.device | None
    d_k: int
    d_v: int
    d_model: int
    num_heads: int
    weights_qkv: Linear # (hd_k + hd_k + hd_v, d_model)
    weights_o: Linear # (d_model, num_heads * d_v) = (d_model, hd_v)
    rope: RotaryPositionalEmbedding | None

    def __init__(self, d_model: int, num_heads: int, max_seq_len: int | None = None, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        if d_model % num_heads != 0:
            raise Exception("d_model must be divisible by num_heads!")
        self.device = device
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = round(d_model / num_heads)
        self.d_v = self.d_k
        self.weights_qkv = Linear(d_model, num_heads * self.d_k + num_heads * self.d_k + num_heads * self.d_v, device, dtype)
        self.weights_o = Linear(num_heads * self.d_v, d_model, device, dtype)
        if max_seq_len is not None and theta is not None:
            self.rope = RotaryPositionalEmbedding(theta, self.d_k, max_seq_len)
        else:
            self.rope = None

    # x (..., seq_len, d_model)
    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        seq_len = x.shape[-2]
        casual_mask = (torch.tril(torch.ones(seq_len, seq_len, device=self.device)) == 1)
        if token_positions is None:
            token_positions = torch.arange(seq_len, device=self.device)
        w_x = self.weights_qkv.forward(x)  # (..., seq_len, hd_k + hd_k + hd_v)
        wq_x_i = rearrange(wq_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wk_x_i = rearrange(wk_x, "... seq_len (h d_k) -> ... h seq_len d_k", h=self.num_heads)  # (..., h, seq_len, d_k)
        wv_x_i = rearrange(wv_x, "... seq_len (h d_v) -> ... h seq_len d_v", h=self.num_heads)  # (..., h, seq_len, d_v)
        if self.rope:
            wq_x_i = self.rope.forward(wq_x_i, token_positions)
            wk_x_i = self.rope.forward(wk_x_i, token_positions)
        result = scaled_dot_product_attention(wq_x_i, wk_x_i, wv_x_i, casual_mask)  # (..., h, seq_len, d_v)
        result = rearrange(result, "... h seq_len d_v -> ... seq_len (h d_v)")
        return self.weights_o.forward(result)


In [477]:
d_model = 16
num_heads = 4
max_seq_len = 8
theta = 10000
attention = CasualMultiHeadSelfAttention(d_model, num_heads)
print(attention)
attention_with_rope = CasualMultiHeadSelfAttention(d_model, num_heads, max_seq_len, theta)
print(attention_with_rope)
x = torch.randn(max_seq_len, d_model)
print(attention.forward(x))
print(attention_with_rope.forward(x))

CasualMultiHeadSelfAttention(
  (weights_q): Linear()
  (weights_k): Linear()
  (weights_v): Linear()
  (weights_o): Linear()
)
CasualMultiHeadSelfAttention(
  (weights_q): Linear()
  (weights_k): Linear()
  (weights_v): Linear()
  (weights_o): Linear()
  (rope): RotaryPositionalEmbedding()
)
tensor([[-9.1347e-01, -3.2879e-01,  2.9832e-01,  1.0010e+00,  6.5051e-02,
         -3.1182e-01,  7.2007e-02,  2.4565e-01,  5.6017e-01, -3.7121e-01,
         -2.1364e-01, -2.5388e-01,  9.7937e-02, -2.5788e-01, -1.0614e-01,
         -3.7475e-01],
        [-8.9447e-01, -1.0819e-01,  9.6417e-03,  1.3280e+00, -8.4503e-01,
         -2.5851e-01, -4.3465e-01,  1.0073e+00, -2.8206e-02,  1.4128e+00,
         -8.1246e-02, -3.9602e-01,  9.0353e-01,  6.7310e-03,  5.0838e-01,
          1.2974e-03],
        [-4.0334e-01,  5.4240e-01, -2.4756e-01,  8.7136e-02, -4.8301e-01,
         -3.2318e-04,  6.7881e-01,  3.3932e-01,  4.2408e-01,  1.3075e+00,
         -7.0993e-01,  4.0007e-01,  8.0679e-01, -4.6665e-01,  2.3595

In [557]:
import torch

class TransformerBlock(torch.nn.Module):

    norm_attention: RMSNorm
    attention: CasualMultiHeadSelfAttention

    feed_forward: SwiGLU
    norm_feed_forward: RMSNorm

    def __init__(self, d_model: int, num_heads: int, d_ff: int | None = None, eps: float = 1e-5, max_seq_len: int | None = None, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.norm_attention = RMSNorm(d_model, eps, device=device, dtype=dtype)
        self.attention = CasualMultiHeadSelfAttention(d_model, num_heads, max_seq_len, theta, device=device, dtype=dtype)
        self.norm_feed_forward = RMSNorm(d_model, eps, device=device, dtype=dtype)
        self.feed_forward = SwiGLU(d_model, d_ff, device=device, dtype=dtype)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        y = x + self.attention(self.norm_attention(x), token_positions)
        y = y + self.feed_forward(self.norm_feed_forward(y))
        return y

In [538]:
d_model = 16
num_heads = 4
d_ff = round(8 * d_model / 3 / 64) * 64
transformer = TransformerBlock(d_model, num_heads, d_ff)
print(transformer)
x = torch.randn(4, d_model)
transformer(x)

TransformerBlock(
  (norm_attention): RMSNorm()
  (attention): CasualMultiHeadSelfAttention(
    (weights_q): Linear()
    (weights_k): Linear()
    (weights_v): Linear()
    (weights_o): Linear()
  )
  (norm_feed_forward): RMSNorm()
  (feed_forward): SwiGLU(
    (w_1): Linear()
    (w_2): Linear()
    (w_3): Linear()
  )
)


tensor([[-1.5405, -2.2231, -2.4360, -1.3539,  0.6168, -0.7161, -0.8907,  0.3413,
          1.9047,  0.3220, -0.4751,  0.0729,  2.0809,  2.3151, -0.5388,  0.9782],
        [-1.5692, -0.8923, -2.1495,  0.2671,  0.2821,  1.4545, -1.3060, -0.8964,
          0.3914, -3.0885,  0.1149,  1.4239, -0.2521,  1.4806, -2.7316,  1.0261],
        [-0.5559,  1.2420,  0.5974, -1.1369,  0.8918, -0.2537,  1.8052, -1.3066,
          2.7895, -1.6797,  0.3634, -0.2439, -1.3328,  0.9814, -1.6988,  1.1178],
        [ 1.2810,  0.2864, -1.0651, -1.7830, -2.0356,  0.4119, -1.0809, -0.9212,
          1.0796,  0.4117,  0.6029,  1.1449, -0.1938,  0.7363, -0.8468,  0.8697]],
       grad_fn=<AddBackward0>)

In [584]:
import torch

class TransformerLM(torch.nn.Module):

    token_embeddings: Embedding
    layers: torch.nn.ModuleList
    ln_final: RMSNorm
    lm_head: Linear

    def __init__(self, vocab_size: int, context_length: int, num_layers: int, d_model: int, num_heads: int, d_ff: int | None = None, eps: float = 1e-5, theta: float | None = None, device: torch.device | None = None, dtype: torch.dtype | None = None):
        super().__init__()
        self.token_embeddings = Embedding(num_embeddings=vocab_size, embedding_dim=d_model, device=device, dtype=dtype)
        self.layers = torch.nn.ModuleList(TransformerBlock(d_model, num_heads, d_ff, eps, context_length, theta, device, dtype) for _ in range(num_layers))
        self.ln_final = RMSNorm(d_model, eps, device=device, dtype=dtype)
        self.lm_head = Linear(d_model, vocab_size, device=device, dtype=dtype)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.token_embeddings(x)
        for transformer in self.layers:
            y = transformer(y)
        y = self.ln_final(y)
        y = self.lm_head(y)
        y = softmax(y, -1)
        return y

In [585]:
vocab_size = 256
context_length = 1000
num_layers = 2
d_model = 16
num_heads = 4
d_ff = round(8 * d_model / 3 / 64) * 64
eps: float = 1e-5
theta = 10000

transformer_lm = TransformerLM(vocab_size, context_length, num_layers, d_model, num_heads, d_ff, eps, theta)
print(transformer_lm)

x = torch.randint(0, 255, (4, 6))
y = transformer_lm(x)
print(y)

TransformerLM(
  (token_embeddings): Embedding()
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (norm_attention): RMSNorm()
      (attention): CasualMultiHeadSelfAttention(
        (weights_q): Linear()
        (weights_k): Linear()
        (weights_v): Linear()
        (weights_o): Linear()
        (rope): RotaryPositionalEmbedding()
      )
      (norm_feed_forward): RMSNorm()
      (feed_forward): SwiGLU(
        (w_1): Linear()
        (w_2): Linear()
        (w_3): Linear()
      )
    )
  )
  (ln_final): RMSNorm()
  (lm_head): Linear()
)
tensor([[[0.0039, 0.0048, 0.0047,  ..., 0.0023, 0.0053, 0.0035],
         [0.0024, 0.0047, 0.0029,  ..., 0.0040, 0.0039, 0.0027],
         [0.0032, 0.0042, 0.0037,  ..., 0.0050, 0.0031, 0.0027],
         [0.0045, 0.0051, 0.0054,  ..., 0.0037, 0.0042, 0.0028],
         [0.0025, 0.0032, 0.0030,  ..., 0.0061, 0.0034, 0.0034],
         [0.0054, 0.0065, 0.0027,  ..., 0.0031, 0.0027, 0.0025]],

        [[0.0036, 0.0052, 0.0041,  ..., 0.